# Cybersecurity Attack Classifier (DistilBERT Fine-Tuning)
This notebook fine-tunes `distilbert-base-uncased` to classify incoming cybersecurity requests, payloads, logs, SQL queries, URLs, and user inputs into 18 attack categories vs. benign traffic.

### Features:
- Base model: `distilbert-base-uncased` (Multi-Class Sequence Classification)
- Automated evaluation: Accuracy, Precision, Recall, F1-score
- CPU Optimization: ONNX Runtime export + PyTorch INT8 Dynamic Quantization for fast VPS deployment
- Model Export: Saves `trained_model/` & compresses to `attack_model.zip`

In [ ]:
# Step 1: Install required dependencies
!pip install -q transformers datasets torch accelerate scikit-learn onnxruntime optimum

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

In [ ]:
# Step 2: Load or Auto-Generate Dataset (Handles Missing File Automatically)
DATASET_PATH = 'attack_dataset.csv'

if not os.path.exists(DATASET_PATH):
    print(f"'{DATASET_PATH}' not found in current directory. Attempting file upload...")
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception as e:
        print("Upload failed or not running in Colab GUI. Generating dataset automatically...")
        raw_data = [
            ("' OR 1=1 --", "SQL_Injection"), ("' OR '1'='1", "SQL_Injection"), ("1; DROP TABLE users; --", "SQL_Injection"),
            ("<script>alert(1)</script>", "XSS"), ("<img src=x onerror=alert(document.cookie)>", "XSS"),
            ("../../etc/passwd", "PathTraversal"), ("../../../../etc/shadow", "PathTraversal"),
            ("; cat /etc/passwd", "Command_Injection"), ("| id", "Command_Injection"),
            ("<?php system($_GET['cmd']); ?>", "RCE"), ("eval(base64_decode('...'))", "RCE"),
            ("<form action='http://bank.com/transfer'>...", "CSRF"),
            ("http://169.254.169.254/latest/meta-data/", "SSRF"), ("gopher://127.0.0.1:6379/_flushall", "SSRF"),
            ("<!DOCTYPE foo [<!ENTITY xxe SYSTEM 'file:///etc/passwd'>]>", "XXE"),
            ("*(|(objectclass=*))", "LDAP_Injection"), ("(&(user=admin)(pass=*))", "LDAP_Injection"),
            ("{\"username\": {\"$gt\": \"\"}}", "NoSQL_Injection"),
            ("shell.php.png", "FileUpload_Attack"), ("malicious.php%00.jpg", "FileUpload_Attack"),
            ("POST /login admin admin123", "BruteForce"), ("POST /auth/login user password123", "BruteForce"),
            ("POST /login_check _username=u1&_password=p1", "CredentialStuffing"),
            ("POST /slowloris HTTP/1.1", "DDoS"),
            ("powershell -enc JABzAD0...", "Malware"), ("cmd.exe /c evil.ps1", "Malware"),
            ("GET / HTTP/1.1\r\nUser-Agent: sqlmap/1.5.2", "Malicious_HTTP"),
            ("GET /?id=%00 HTTP/1.1", "Suspicious_Input"),
            ("GET /api/v1/products?category=electronics HTTP/1.1", "Benign"),
            ("POST /api/v1/contact {\"name\":\"John\"}", "Benign")
        ]
        df_fallback = pd.DataFrame(raw_data, columns=['text', 'label'])
        df_fallback.to_csv(DATASET_PATH, index=False)

df = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully! Shape:", df.shape)
print(df.head())

# Create Label Mappings
labels = sorted(df['label'].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
num_labels = len(labels)

df['label_id'] = df['label'].map(label2id)
print(f"Loaded {num_labels} distinct target labels: {labels}")

In [ ]:
# Step 3: Tokenize Dataset using distilbert-base-uncased
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

dataset = Dataset.from_pandas(df[['text', 'label_id']].rename(columns={'label_id': 'label'}))
dataset = dataset.train_test_split(test_size=0.2, seed=42)

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_dataset = dataset.map(preprocess_function, batched=True)
print("Tokenized dataset sample:", tokenized_dataset['train'][0])

In [ ]:
# Step 4: Load DistilBERT Model for Multi-Class Sequence Classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
# Step 5: Define Evaluation Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
# Step 6: Fine-Tune the Model (Updated for Transformers 4.46+ / 5.0+ compat)
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=10,
    report_to='none'
)

# In Transformers 4.46+, processing_class replaces tokenizer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# Step 7: Evaluate Model Performance
eval_results = trainer.evaluate()
print("\n--- Final Evaluation Metrics ---")
for k, v in eval_results.items():
    print(f"{k}: {v:.4f}")

# Detailed Classification Report
predictions = trainer.predict(tokenized_dataset['test'])
preds = np.argmax(predictions.predictions, axis=-1)
print("\nClassification Report:")
print(classification_report(tokenized_dataset['test']['label'], preds, target_names=[id2label[i] for i in range(num_labels)]))

In [ ]:
# Step 8: Save Model & Tokenizer Files
SAVE_DIR = './trained_model'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model and tokenizer saved to {SAVE_DIR}")

In [ ]:
# Step 9: Export INT8 Dynamic Quantization for Ultra-Fast CPU Inference
import torch.quantization

quantized_model = torch.quantization.quantize_dynamic(
    model.cpu(),
    {torch.nn.Linear},
    dtype=torch.qint8
)
torch.save(quantized_model.state_dict(), os.path.join(SAVE_DIR, 'quantized_model.pt'))
print("Quantized PyTorch CPU model saved.")

In [ ]:
# Step 10: Compress Model Artifacts for Download
!zip -r attack_model.zip trained_model/ attack_dataset.csv attack_dataset.json
print("Successfully created attack_model.zip!")